# Казахско-китайский корпус: сборка с сохранением на Google Drive

**Как пользоваться**
1. Среда выполнения → Сменить тип среды → **GPU** (иначе LaBSE будет очень медленным).
2. Среда выполнения → **Выполнить все**. При первом запуске Google попросит доступ к Диску.
3. Результат лежит на Диске: в папке `BASE` (см. ячейку 1), подпапка **corpus_v0.5 /** (обычная папка, не архив).

**Повторный запуск ничего не пересчитывает:**
- скачанные данные OPUS лежат в `kkzh_corpus/opus/` и повторно не качаются;
- оценки LaBSE кэшируются в `kkzh_corpus/labse_scores.tsv` (при обрыве сессии расчёт продолжится с места остановки, а смена порога `TAU` занимает секунды);
- если папка версии уже собрана, сборка пропускается. Чтобы собрать заново с другими настройками, поменяйте `VERSION` (новая папка рядом со старой) или поставьте `OVERWRITE = True`.


In [ ]:
# 1. Подключение Google Drive и настройки
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, subprocess, shutil, hashlib

BASE = '/content/drive/MyDrive/kkzh_corpus'   # папка на Диске: тут хранится всё (скачанный OPUS и кэш LaBSE уже лежат здесь)
VERSION = 'corpus_v0.13'                       # другая версия = другая папка рядом
USE_WIKI = True                               # добавить пары из Википедии (kk-zh), лицензия CC BY-SA 4.0
RUN_DIAG = False                              # True: подготовить выборки для ручной проверки (ячейка 7)
OVERWRITE = False                             # True: пересобрать, даже если папка версии уже есть
FORCE_DOWNLOAD = False                        # True: скачать OPUS заново

# Параметры сборки
ADD_LOCALIZATION = True                      # True: добавить наборы OPUS по локализации ПО (KDE4, GNOME, Ubuntu): короткие строки интерфейса
ADD_TRADITIONAL = False                       # True: взять и варианты с традиционными иероглифами (zh_TW, zh_HK); они переводятся в упрощённые, дубликаты уберутся
SKIP_DOWNLOAD = 'XLEnt' if ADD_LOCALIZATION else 'KDE4,GNOME,Ubuntu,XLEnt'   # XLEnt (списки сущностей) не берём в любом случае
HELDOUT = 'opus:TED2020,opus:News-Commentary' # dev/test берутся только отсюда
N_DEV, N_TEST = 1000, 1000
TAU = 0.72                                    # порог LaBSE
WIKI_MIN_COS = 0.72                          # Википедия: пары с оценкой майнера ниже этого в сборку не берём (они всё равно не пройдут порог; экономит часы расчёта LaBSE)
WIKI_TAU = 0.72                              # порог LaBSE только для пар из Википедии (OPUS использует TAU)
TEMPLATE_CAP = 300                            # Википедия: не больше стольких предложений одного шаблона («X — коммуна в Италии…»); 0 = без ограничения
ZH_SCRIPT = 'to_simplified'                   # традиционные иероглифы -> упрощённые (письменный кантонский всегда отбрасывается); 'drop_traditional' / 'keep'
EXTRA_BUILD_ARGS = []                         # например ['--ratio-min', '0.2']
# Источники, которые убираются из train: dev/test берутся из HELDOUT, и рядом с ними в train не должно
# остаться кусков тех же докладов/статей (в OPUS граница документа известна лишь с точностью до 50 строк).
EXCLUDE_FROM_TRAIN = 'opus:TED2020,opus:News-Commentary,opus:NeuLab-TedTalks,opus:QED'

OPUS_DIR = f'{BASE}/opus'
OPUS_ALL = f'{OPUS_DIR}/opus_all.tsv'
USE_WEBLATE = True                           # True: добавить пары из Weblate-Translations (kk и zh связаны через английскую строку); строки интерфейсов, у каждой строки своя лицензия
WEBLATE_TRADITIONAL = False                   # True: брать и традиционные иероглифы (потом переводятся в упрощённые)
WEBLATE_DIR = f'{BASE}/weblate'
WIKI_DIR = f'{BASE}/wiki'
WIKI_CONTACT = ''                             # по желанию: ваша почта; Википедия просит указывать контакт, тогда она реже ограничивает запросы
WIKI_EXTRA_ARGS = []                          # служебное: например ['--min-cos', '0.6']
ALL_SRC = f'{BASE}/all_sources.tsv'            # общий входной файл: OPUS (+ Википедия)
OUT = f'{BASE}/{VERSION}'
CACHE = f'{BASE}/labse_scores.tsv'
os.makedirs(BASE, exist_ok=True)


def sh(cmd, log=None):
    """Запустить команду, показывать вывод сразу и (по желанию) писать его в файл."""
    print('$', ' '.join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    f = open(log, 'w', encoding='utf-8') if log else None
    try:
        for line in p.stdout:
            print(line, end='')
            if f:
                f.write(line)
    finally:
        p.wait()
        if f:
            f.close()
    if p.returncode != 0:
        raise RuntimeError(f'команда завершилась с кодом {p.returncode}')

print('Папка на Диске:', BASE)


In [ ]:
# 2. Зависимости
sh([sys.executable, '-m', 'pip', 'install', '-q',
    'datasketch', 'opencc-python-reimplemented', 'sentence-transformers', 'datasets'])
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else 'НЕТ. Включите GPU в типе среды, иначе LaBSE будет очень медленным')
except ImportError:
    print('torch не найден')


Скрипты вшиты в ноутбук: две следующие ячейки просто записывают их в `/content` (код не менялся, это те же `fetch_opus.py` и `build_corpus.py`).

In [ ]:
%%writefile /content/fetch_opus.py
#!/usr/bin/env python3
"""
List / download Kazakh-Chinese bitexts from OPUS and convert them to the TSV
format used by build_corpus.py.  Needs internet access, standard library only.

  python fetch_opus.py list     --out opus_listing.json
  python fetch_opus.py download --out opus/ [--only A,B] [--skip C] [--max-pairs 300000]

NOTE: written against the public OPUS API (https://opus.nlpl.eu/opusapi/) but
NOT tested against the live service. Run `list` first and look at the output;
if the JSON keys differ, use --debug and adjust ITEM_KEYS below.
"""
import argparse
import csv
import io
import json
import os
import re
import sys
import unicodedata
import urllib.parse
import urllib.request
import zipfile

API = "https://opus.nlpl.eu/opusapi/"
# Chinese language-code variants used in OPUS (simplified / traditional / generic)
ZH_CODES = ["zh", "zh_CN", "zh_cn", "zh_TW", "zh_tw", "zh_HK", "zho"]
TRADITIONAL = {"zh_tw", "zh_hk", "zh_mo"}
ITEM_KEYS = dict(corpus="corpus", url="url", source="source", target="target",
                 pairs="alignment_pairs", version="version")
WS = re.compile(r"\s+")
CTRL = re.compile(r"[\u0000-\u0008\u000b-\u001f\u007f\u200b\u200e\u200f\u202a-\u202e\ufeff]")


def get(url, binary=False, timeout=300):
    req = urllib.request.Request(url, headers={"User-Agent": "kkzh-corpus-builder/0.1"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        data = r.read()
    return data if binary else json.loads(data.decode("utf-8"))


def list_corpora(api, debug=False):
    found = {}
    for z in ZH_CODES:
        for s, t in (("kk", z), (z, "kk")):
            q = api + "?" + urllib.parse.urlencode(
                dict(source=s, target=t, preprocessing="moses", version="latest"))
            try:
                data = get(q)
            except Exception as e:  # network or HTTP error for this variant
                print(f"  [warn] {s}-{t}: {e}", file=sys.stderr)
                continue
            items = data.get("corpora", []) if isinstance(data, dict) else data
            if debug and items:
                print("RAW ITEM:", json.dumps(items[0], ensure_ascii=False)[:500])
            for it in items:
                url = it.get(ITEM_KEYS["url"])
                if url:
                    found[url] = it
    return list(found.values())


def clean(s):
    return WS.sub(" ", CTRL.sub("", unicodedata.normalize("NFC", s))).strip()


def pair_files(zf, kk_code, zh_code):
    names = zf.namelist()
    kk = [n for n in names if n.endswith("." + kk_code)]
    zh = [n for n in names if n.endswith("." + zh_code)]
    return (kk[0], zh[0]) if kk and zh else (None, None)


def convert(item, zbytes, max_pairs):
    a, b = item[ITEM_KEYS["source"]], item[ITEM_KEYS["target"]]
    kk_code, zh_code = (a, b) if a.startswith("kk") else (b, a)
    zf = zipfile.ZipFile(io.BytesIO(zbytes))
    fk, fz = pair_files(zf, kk_code, zh_code)
    if not fk:
        print(f"  [skip] no Moses files for {kk_code}/{zh_code}: {zf.namelist()[:8]}", file=sys.stderr)
        return []
    name = item[ITEM_KEYS["corpus"]]
    rows = []
    with zf.open(fk) as k, zf.open(fz) as z:
        for i, (lk, lz) in enumerate(zip(io.TextIOWrapper(k, encoding="utf-8"),
                                         io.TextIOWrapper(z, encoding="utf-8"))):
            if max_pairs and i >= max_pairs:
                break
            rows.append((clean(lk), clean(lz), f"opus:{name}", f"opus:{name}#chunk{i // 50}",
                         f"CHECK-LICENCE (OPUS corpus {name}; zh code {zh_code})",
                         item.get(ITEM_KEYS["url"], "")))
    return rows


def write_tsv(path, rows):
    with open(path, "w", encoding="utf-8", newline="") as f:
        # no quoting and no escape character: fields are written verbatim (the old escapechar put a
        # backslash before every double quote, and each re-read/re-write added one more)
        w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_NONE, quotechar=None,
                       lineterminator="\n")
        w.writerow(["kk", "zh", "source", "doc_id", "license", "url"])
        w.writerows([[" ".join(str(v).split()) for v in row] for row in rows])


def main(argv=None):
    if argv is None and any(k in sys.argv[0] for k in ("ipykernel", "colab_kernel_launcher")):
        print("Notebook detected: sys.argv belongs to the Jupyter kernel, not to this script.\n"
              "Either save this file and run it as a command, e.g.\n"
              "    !python fetch_opus.py <command> --options\n"
              "or call it from a cell with an explicit argument list, e.g.\n"
              "    main(['list', '--out', 'opus/listing.json'])")
        return

    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("cmd", choices=["list", "download"])
    ap.add_argument("--api", default=API)
    ap.add_argument("--out", required=True)
    ap.add_argument("--only", default="", help="comma-separated: NAME or NAME:zhcode, e.g. KDE4:zh_CN")
    ap.add_argument("--skip", default="", help="comma-separated: NAME or NAME:zhcode, e.g. KDE4:zh_CN")
    ap.add_argument("--no-traditional", action="store_true",
                    help="skip zh_TW / zh_HK variants (they mostly duplicate the zh_CN variants)")
    ap.add_argument("--max-pairs", type=int, default=0, help="cap per corpus (0 = no cap)")
    ap.add_argument("--debug", action="store_true")
    a = ap.parse_args(argv)

    items = list_corpora(a.api, a.debug)
    if not items:
        sys.exit("API returned no kk-zh corpora (or the service is unreachable).")
    print(f"{'corpus':<22}{'pairs':>12}  {'langs':<14} url")
    for it in sorted(items, key=lambda x: -int(x.get(ITEM_KEYS["pairs"]) or 0)):
        print(f"{it.get(ITEM_KEYS['corpus'], '?'):<22}{it.get(ITEM_KEYS['pairs'], '?'):>12}  "
              f"{it.get(ITEM_KEYS['source'], '?')}-{it.get(ITEM_KEYS['target'], '?'):<10} "
              f"{it.get(ITEM_KEYS['url'], '')}")
    if a.cmd == "list":
        os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
        json.dump(items, open(a.out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        print(f"\nlisting saved to {a.out}")
        return

    only = {x for x in a.only.split(",") if x}
    skip = {x for x in a.skip.split(",") if x}
    os.makedirs(a.out, exist_ok=True)
    allrows = []
    failed = []
    for it in items:
        name = it.get(ITEM_KEYS["corpus"])
        sa, sb = it[ITEM_KEYS["source"]], it[ITEM_KEYS["target"]]
        zh_code = sb if sa.startswith("kk") else sa
        keys = {name, f"{name}:{zh_code}"}          # e.g. "KDE4" or "KDE4:zh_CN"
        if a.no_traditional and zh_code.lower() in TRADITIONAL:
            continue
        if (only and not (keys & only)) or (keys & skip):
            continue
        print(f"downloading {name} ({zh_code}) ...")
        try:
            rows = convert(it, get(it[ITEM_KEYS["url"]], binary=True), a.max_pairs)
        except Exception as e:
            print(f"  [error] {name} ({zh_code}): {e}", file=sys.stderr)
            failed.append(f"{name}:{zh_code}")
            continue
        write_tsv(os.path.join(a.out, f"{name}.{zh_code}.tsv"), rows)
        print(f"  {len(rows)} pairs")
        allrows.extend(rows)
    if failed:
        sys.exit(f"\nDownload INCOMPLETE, failed: {', '.join(failed)}. opus_all.tsv was NOT written "
                 f"(so that an incomplete file is never mistaken for a finished one). Rerun.")
    if not allrows:
        sys.exit("\nNothing was selected/downloaded (check --only / --skip / --no-traditional).")
    write_tsv(os.path.join(a.out, "opus_all.tsv"), allrows)
    print(f"\n{len(allrows)} pairs total -> {os.path.join(a.out, 'opus_all.tsv')}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/build_corpus.py
#!/usr/bin/env python3
"""
Kazakh-Chinese parallel corpus builder.

Pipeline for collecting, filtering and splitting Kazakh–Chinese pairs.

Input : TSV with header  kk <TAB> zh [<TAB> source <TAB> doc_id <TAB> license <TAB> url]
Output: train.tsv / dev.tsv / test.tsv, manifest.json, CHECKSUMS.sha256

Subcommands
  moses2tsv   convert two line-aligned text files (OPUS "Moses" format) to TSV
  build       normalise, filter, (optionally) LaBSE-score, dedup, split, hash
  audit       draw a stratified sample for manual annotation
  kappa       Cohen's kappa + precision from the filled annotation sheet
"""
import argparse
import collections
import csv
import datetime
import hashlib
import json
import math
import os
import random
import re
import sys
import unicodedata

csv.field_size_limit(10**8)

# ----------------------------------------------------------------- script tests
HAN = re.compile(r"[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]")
CYR = re.compile(r"[\u0400-\u04ff]")
ARAB = re.compile(r"[\u0600-\u06ff\u0750-\u077f\ufb50-\ufdff\ufe70-\ufeff]")
LAT = re.compile(r"[A-Za-z\u00c0-\u024f\u0131]")
KK_CYR = set("әіңғүұқөһӘІҢҒҮҰҚӨҺ")
JUNK = re.compile(r"(https?://|www\.|@\w+\.\w+)")
CTRL = re.compile(r"[\u0000-\u0008\u000b-\u001f\u007f\u200b\u200e\u200f\u202a-\u202e\ufeff]")
WS = re.compile(r"\s+")
BS_QUOTE = re.compile(r'\\+(?=")')                        # stray backslashes before a quote (artefact of an old TSV writer)
ASS_TAG = re.compile(r"\{\\[^}]*\}")              # subtitle override tags such as {\fn..\fs12}
KK_LAT_DIAC = re.compile("[\u00e1\u00c1\u01f5\u01f4\u0131\u0130\u0144\u0143\u00f3\u00d3\u00fa\u00da"
                         "\u00fd\u00dd\u00e4\u00c4\u00f6\u00d6\u00fc\u00dc\u00f1\u00d1\u011f\u011e"
                         "\u015f\u015e\u00e7\u00c7]")                    # letters of the Kazakh Latin alphabets
# Cyrillic letters of the Serbian/Macedonian block never occur in Russian or Kazakh text; they appear when
# Arabic-script subtitles (cp1256 bytes) were decoded as cp1251 ("mojibake").
MOJIBAKE_LETTERS = re.compile("[\u0403\u0453\u0409\u040a\u040c\u040b\u040f\u0452\u0459\u045a\u045c"
                              "\u045b\u045f\u040e\u045e\u0408\u0458\u0405\u0455]")
MOJIBAKE_INWORD = re.compile(r'(?<=[\u0400-\u04ff])\\*["\'\u00ac\u00f7\u2014]+(?=[\u0400-\u04ff])')
# Russian function words that are NOT common Kazakh words (used to detect Russian / code-switched text)
RU_STOP = set(("и в во что он я как то все она так его но ты за только ее мне было вот от меня еще нет из "
               "ему теперь когда даже вдруг если уже или быть был него до вас нибудь опять уж вам ведь потом "
               "себя ничего ей может они где есть надо ней для тебя их чем была чтоб без будто чего тоже себе "
               "под будет тогда кто этот того потому этого какой совсем ним здесь этом почти тем чтобы нее "
               "сейчас были куда зачем всех никогда можно при наконец об другой после над больше через эти нас "
               "про всего них какая много разве эту моя впрочем хорошо свою этой перед иногда лучше том нельзя "
               "такой им более всегда конечно всю между это").split()) \
    - {"ей", "ней", "ним", "ему", "вам", "вас", "нас", "нее"}
RU_TOKEN = re.compile("[\u0430-\u044f\u0451\u04d9\u0456\u04a3\u0493\u04af\u04b1\u049b\u04e9\u04bb]+")
CANTONESE = re.compile("[\u5605\u5497\u4f62\u54cb\u5570\u5190\u55ba\u5514\u5481\u5622\u569f\u5571"
                       "\u5625\u755b\u775c\u8af7\u4e5c\u54a9]")        # written-Cantonese-only characters
SPAM_KK = re.compile(r"казино|слот|1xbet|порно|секс|эротик|лесбиян|проститут|интим", re.I)
SPAM_ZH = re.compile("\u8d4c\u573a|\u8d4c\u535a|\u5a31\u4e50\u573a|\u8001\u864e\u673a|\u535a\u5f69|"
                     "\u767e\u5bb6\u4e50|\u8272\u60c5|\u6027\u7231|\u505a\u7231|\u88f8\u4f53|"
                     "\u53e3\u4ea4|\u809b\u4ea4|\u809b\u95e8|\u4e71\u4f26|\u6210\u4eba\u89c6\u9891|"
                     "\u60c5\u8272|\u7ea6\u70ae|1xbet", re.I)


def normalise(s):
    s = unicodedata.normalize("NFC", s)
    s = CTRL.sub("", s)
    s = ASS_TAG.sub("", s)
    s = BS_QUOTE.sub("", s)
    return WS.sub(" ", s).strip()


def kk_script(s):
    c, a, l = len(CYR.findall(s)), len(ARAB.findall(s)), len(LAT.findall(s))
    tot = c + a + l
    if tot == 0:
        return None, 0.0
    best = max((c, "cyr"), (a, "arab"), (l, "lat"))
    return best[1], best[0] / tot


def zh_chars(s):
    return len(re.sub(r"\s", "", s))


def zh_other(zh):
    """(number of Han characters, share of Cyrillic/Arabic/Latin letters among Han+those letters)."""
    n_han = len(HAN.findall(zh))
    other = len(CYR.findall(zh)) + len(ARAB.findall(zh)) + len(LAT.findall(zh))
    return n_han, (other / (n_han + other) if n_han + other else 0.0)


_EN_RUN = re.compile(r"[A-Za-z][A-Za-z0-9'\u2019\-]*(?:[ ,.!?;:\"\u2026\-]+[A-Za-z0-9'\u2019\-]+)+[.!?\u2026\"]*")


def strip_zh_english(zh):
    """OpenSubtitles zh lines often carry the English subtitle line too ("好吧 你赢了 Okay, you win.").
    Remove every run of 2+ English words."""
    out = _EN_RUN.sub(lambda m: "" if len(re.findall(r"[A-Za-z]+", m.group(0))) >= 2 else m.group(0), zh)
    return WS.sub(" ", out).strip(" -")


def ru_hits(kk):
    toks = RU_TOKEN.findall(kk.lower())
    return sum(t in RU_STOP for t in toks), len(toks)


def kk_mojibake(kk):
    return bool(MOJIBAKE_LETTERS.search(kk)) or len(MOJIBAKE_INWORD.findall(kk)) >= 2


_T2S = None


def _t2s():
    global _T2S
    if _T2S is None:
        try:
            from opencc import OpenCC
            _T2S = OpenCC("t2s")
        except ImportError:
            _T2S = False
            print("WARNING: opencc not installed (pip install opencc-python-reimplemented); "
                  "traditional/simplified handling is DISABLED", file=sys.stderr)
    return _T2S


def handle_zh_script(zh, mode, thr=0.03):
    """Return (zh, reason). Traditional text is detected by how much OpenCC t2s changes it."""
    if mode == "keep":
        return zh, None
    if CANTONESE.search(zh):          # written Cantonese is not Mandarin: converting the script does not fix it
        return zh, "zh_cantonese"
    if not _t2s():
        return zh, None
    simp = _T2S.convert(zh)
    han = [(x, y) for x, y in zip(zh, simp) if HAN.match(x)]
    share = sum(x != y for x, y in han) / len(han) if han else 0.0
    if mode == "to_simplified":
        return simp, None
    return zh, ("zh_traditional" if share > thr else None)


# ------------------------------------------------------------------- filtering
# ---- extra rules for comparable (mined) sources such as Wikipedia ----------
_NUM = re.compile(r"\d+(?:[.,]\d+)*")
_TERMINAL = tuple(".!?\u2026\u3002\uff01\uff1f\"\u00bb\u201d\u300d\u300f)\uff09")


def numbers(s):
    """Set of digit strings in s (thousand separators / decimal marks removed)."""
    s = re.sub(r"(?<=\d)[\s\u00a0\u202f](?=\d{3}(?!\d))", "", s)
    return {m.replace(",", "").replace(".", "") for m in _NUM.findall(s)}


def strict_reason(kk, zh):
    """Rules for mined sources. Article pairs are comparable, not parallel, so
    the sentence may be a good match in meaning yet differ in a number or date."""
    nk, nz = numbers(kk), numbers(zh)
    if nk - nz:                                   # a Kazakh number is missing on the Chinese side
        return "number_mismatch"
    kw = len(kk.split())
    if kw <= 8 and not kk.rstrip().endswith(_TERMINAL):
        return "title_like"                       # category names, headings, list items
    if kk.rstrip().endswith(":"):
        return "title_like"
    return None


_UP = re.compile(r"^[A-ZА-ЯЁӘІҢҒҮҰҚӨҺ]")


def template_key(kk):
    """Mask names and digits; sentences that differ only in names share a key."""
    kk = re.sub(r"\([^)]*\)", "()", kk)
    toks = []
    for t in kk.split():
        t = t.strip(",;")
        if _UP.match(t) or re.search(r"\d", t):
            toks.append("X")
        else:
            toks.append(t.lower())
    if len(toks) < 5 or sum(t != "X" for t in toks) < 3:
        return None
    return " ".join(toks)


# ---- tidy-up of software-localisation strings (KDE4 / GNOME / Ubuntu) ------------------
_CJK = "\u2e80-\u9fff\u3000-\u303f\uff00-\uffef\u2018\u2019\u201c\u201d"


def tidy_ui(kk, zh):
    """OPUS tokenised these strings: '% 1' instead of '%1', spaces inside Chinese text, keyboard
    accelerators such as '(N)' or '&N'. Applied after scoring, so the LaBSE cache is unaffected."""
    kk = re.sub(r"%\s+(\d)", r"%\1", kk)
    kk = re.sub(r"(^|\s)&\s?(?=\w)", r"\1", kk)
    zh = re.sub(r"%\s+(\d)", r"%\1", zh)
    zh = re.sub(r"\s*\(\s*[A-Za-z]\s*\)\s*(?=(\.\.\.|\u2026|$))", "", zh)     # accelerator "(N)" at the end
    zh = re.sub(r"(?<=[%s])\s+(?=[%s])" % (_CJK, _CJK), "", zh)
    zh = re.sub(r"\(\s+", "(", zh)
    zh = re.sub(r"\s+\)", ")", zh)
    zh = re.sub(r"(?<=[%s])\s+(?=[(\[])" % _CJK, "", zh)
    zh = re.sub(r"(?<=[)\]])\s+(?=[%s])" % _CJK, "", zh)
    return " ".join(kk.split()), " ".join(zh.split())


def check_pair(kk, zh, a):
    """Return (script, reason). reason is None if the pair is kept."""
    if not kk or not zh:
        return None, "empty"
    if kk == zh:
        return None, "identical"
    if kk_mojibake(kk):
        return None, "kk_mojibake"
    script, share = kk_script(kk)
    if script is None:
        return None, "kk_no_letters"
    if share < a.kk_script_share:
        return script, "kk_mixed_script"
    if script == "lat" and a.kk_latin != "keep" and \
            (a.kk_latin == "drop" or len(KK_LAT_DIAC.findall(kk)) < 2):
        return script, "kk_latin_not_kazakh"
    n_han, other_share = zh_other(zh)
    if n_han < 2:
        return script, "zh_no_han"
    if other_share > a.zh_nonhan_share:
        return script, "zh_nonhan_share"
    kw, zc = len(kk.split()), zh_chars(zh)
    if not (a.min_kk_words <= kw <= a.max_kk_words):
        return script, "kk_length"
    if not (a.min_zh_chars <= zc <= a.max_zh_chars):
        return script, "zh_length"
    # Length ratio uses Kazakh words divided by Chinese characters.
    ratio = kw / zc
    if not (a.ratio_min <= ratio <= a.ratio_max):
        return script, "length_ratio"
    if JUNK.search(kk) or JUNK.search(zh):
        return script, "url_or_email"
    digits = sum(ch.isdigit() for ch in kk + zh)
    if digits / max(1, len(kk) + len(zh)) > a.max_digit_share:
        return script, "digit_share"
    if re.search(r"(.)\1{9,}", kk) or re.search(r"(.)\1{9,}", zh):
        return script, "char_repeat"
    if script == "cyr" and not a.keep_russian:
        hits, ntok = ru_hits(kk)
        if (hits >= 2 and hits / max(ntok, 1) >= 0.2) or (hits >= 3 and hits / max(ntok, 1) >= 0.15):
            return script, "likely_russian"              # Russian or Russian/Kazakh code-switching
        if kw >= 5 and hits >= 1 and not any(ch in KK_CYR for ch in kk):
            return script, "likely_russian"
    if not a.keep_spam and (SPAM_KK.search(kk) or SPAM_ZH.search(zh)):
        return script, "spam_adult_gambling"
    return script, None


# ---------------------------------------------------------------------- utils
def pair_id(kk, zh):
    return hashlib.sha256((kk + "\t" + zh).encode("utf-8")).hexdigest()[:16]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def read_tsv(path):
    with open(path, encoding="utf-8", newline="") as f:
        rd = csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
        if rd.fieldnames is None or "kk" not in rd.fieldnames or "zh" not in rd.fieldnames:
            sys.exit(f"{path}: header must contain columns 'kk' and 'zh'")
        for row in rd:
            yield row


def write_tsv(path, rows, cols):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_NONE, quotechar=None,
                       lineterminator="\n")
        w.writerow(cols)
        for r in rows:
            # tabs/newlines cannot be represented in a TSV; normalised text has none, other columns might
            w.writerow([" ".join(str(r.get(c, "")).split()) for c in cols])


def percentiles(vals, ps=(1, 5, 25, 50, 75, 95, 99)):
    if not vals:
        return {}
    v = sorted(vals)
    return {f"p{p}": round(v[min(len(v) - 1, int(len(v) * p / 100))], 4) for p in ps}


def shingles(text, n=5):
    return {text[i:i + n] for i in range(max(1, len(text) - n + 1))}


def minhash(text, num_perm=128):
    from datasketch import MinHash
    m = MinHash(num_perm=num_perm)
    for sh in shingles(text):
        m.update(sh.encode("utf-8"))
    return m


def ngram_keys(kk, zh, n=8):
    """Word 8-grams on the Kazakh side, character 8-grams on the Chinese side."""
    out = set()
    w = kk.lower().split()
    for i in range(len(w) - n + 1):
        out.add("k:" + " ".join(w[i:i + n]))
    z = re.sub(r"\s", "", zh)
    for i in range(len(z) - n + 1):
        out.add("z:" + z[i:i + n])
    return out


# ------------------------------------------------------------------- LaBSE
def labse_scores(pairs, model_name, batch, on_chunk=None):
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        sys.exit("pip install sentence-transformers   (needed for --labse)")
    model = SentenceTransformer(model_name)
    scores = []
    for i in range(0, len(pairs), batch):
        chunk = pairs[i:i + batch]
        a = model.encode([p[0] for p in chunk], batch_size=128, normalize_embeddings=True)
        b = model.encode([p[1] for p in chunk], batch_size=128, normalize_embeddings=True)
        part = [float(x @ y) for x, y in zip(a, b)]
        scores.extend(part)
        if on_chunk:
            on_chunk(chunk, part)
        print(f"  LaBSE {min(i + batch, len(pairs))}/{len(pairs)}", file=sys.stderr)
    return scores


def labse_scores_cached(pairs, model_name, batch, cache_path):
    """LaBSE scores with an on-disk cache keyed by the pair hash.

    The cache is appended to after every chunk, so an interrupted run (Colab
    disconnect) resumes where it stopped, and changing --tau or any filter
    afterwards does not trigger recomputation.  Scores depend only on the pair
    and the model, never on the thresholds, which is why this is safe.
    """
    if not cache_path:
        return labse_scores(pairs, model_name, batch)
    cache, header = {}, "#model\t" + model_name
    if os.path.exists(cache_path) and os.path.getsize(cache_path) > 0:
        with open(cache_path, encoding="utf-8") as f:
            first = f.readline().rstrip("\n")
            if first != header:
                sys.exit(f"{cache_path} was written for a different model ({first!r}); "
                         f"delete it or point --labse-cache elsewhere")
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) == 2:           # ignore a truncated last line
                    try:
                        cache[parts[0]] = float(parts[1])
                    except ValueError:
                        pass
    keys = [pair_id(k, z) for k, z in pairs]
    missing = [(k, p) for k, p in zip(keys, pairs) if k not in cache]
    # unique missing pairs only (identical pairs share a key)
    uniq = {}
    for k, p in missing:
        uniq.setdefault(k, p)
    print(f"  LaBSE cache: {len(pairs) - len(missing)} hits, {len(uniq)} to compute",
          file=sys.stderr)
    if uniq:
        new_file = not os.path.exists(cache_path) or os.path.getsize(cache_path) == 0
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        with open(cache_path, "a", encoding="utf-8") as f:
            if new_file:
                f.write(header + "\n")

            def save(chunk, part):
                for (kk, zh), sc in zip(chunk, part):
                    key = pair_id(kk, zh)
                    sc = round(sc, 4)   # round ONCE, so cold and cached runs are bit-identical
                    cache[key] = sc
                    f.write(f"{key}\t{sc:.4f}\n")
                f.flush()
            labse_scores(list(uniq.values()), model_name, batch, on_chunk=save)
    return [cache[k] for k in keys]


# --------------------------------------------------------------------- build
def cmd_build(a):
    stats = collections.OrderedDict()
    drops = collections.Counter()
    rows, ratios = [], []
    rej_rng = random.Random(a.seed)
    rejected = collections.defaultdict(list)       # reason -> reservoir sample of dropped pairs

    def note_reject(reason, source, kk, zh, score=""):
        """Reservoir sampling: keeps a uniform random sample of --dump-k dropped pairs per reason."""
        if not a.dump_rejected:
            return
        res, n = rejected[reason], drops[reason]   # n counts this item already
        item = dict(reason=reason, source=source, labse=score, kk=kk, zh=zh)
        if len(res) < a.dump_k:
            res.append(item)
        else:
            j = rej_rng.randrange(n)
            if j < a.dump_k:
                res[j] = item

    # ---- 1. read, normalise, filter
    n_in = 0
    n_stripped = 0
    strip_src = {x.strip() for x in a.zh_strip_english_sources.split(",") if x.strip()}
    strict_src = {x.strip() for x in a.strict_sources.split(",") if x.strip()}
    for i, r in enumerate(read_tsv(a.input)):
        n_in += 1
        kk, zh = normalise(r.get("kk", "")), normalise(r.get("zh", ""))
        zh, zreason = handle_zh_script(zh, a.zh_script)
        if strip_src and (r.get("source") or "").strip() in strip_src and not zreason:
            n_han, osh = zh_other(zh)
            if n_han >= 2 and osh > a.zh_nonhan_share:      # only rescue lines that would be dropped anyway
                zh2 = strip_zh_english(zh)
                if zh2 != zh:
                    zh = zh2
                    n_stripped += 1
        script, reason = check_pair(kk, zh, a)
        reason = zreason or reason
        if reason is None and strict_src and (r.get("source") or "").strip() in strict_src:
            reason = strict_reason(kk, zh)
        if reason is None or reason == "length_ratio":
            # collected BEFORE the ratio filter so the percentiles help calibration
            ratios.append(len(kk.split()) / zh_chars(zh))
        if reason:
            drops[reason] += 1
            note_reject(reason, (r.get("source") or "unknown").strip(), kk, zh)
            continue
        src = (r.get("source") or "unknown").strip()
        doc = (r.get("doc_id") or "").strip()
        if not doc:  # fall back to sequential chunks of 50 lines per source
            doc = f"{src}#chunk{i // 50}"
        rows.append(dict(kk=kk, zh=zh, script=script, source=src, doc_id=doc,
                         license=(r.get("license") or "").strip(),
                         url=(r.get("url") or "").strip()))
    stats["input_pairs"] = n_in
    stats["after_rule_filters"] = len(rows)
    stats["zh_english_subtitle_stripped"] = n_stripped
    ratio_pct = percentiles(ratios)

    # ---- 2. exact dedup
    seen, uniq = set(), []
    for r in rows:
        k = (r["kk"].lower(), r["zh"])
        if k in seen:
            drops["exact_duplicate"] += 1
            continue
        seen.add(k)
        uniq.append(r)
    rows = uniq
    stats["after_exact_dedup"] = len(rows)

    # ---- 3. LaBSE alignment score
    if a.labse:
        todo = [r for r in rows if not (r["script"] == "arab" and a.arabic_policy != "score")]
        if a.arabic_policy == "drop":
            drops["arabic_script_dropped"] += len(rows) - len(todo)
            rows = todo
        sc = labse_scores_cached([(r["kk"], r["zh"]) for r in todo], a.labse_model,
                                 a.labse_batch, a.labse_cache)
        for r, s in zip(todo, sc):
            r["labse"] = round(s, 4)
        kept = []
        for r in rows:
            thr = a.strict_tau if (a.strict_tau is not None and r["source"] in strict_src) else a.tau
            if "labse" in r and r["labse"] < thr:
                drops["labse_below_tau"] += 1
                note_reject("labse_below_tau", r["source"], r["kk"], r["zh"], r["labse"])
                continue
            kept.append(r)
        rows = kept
        stats["after_labse"] = len(rows)
    else:
        print("WARNING: no --labse: alignment scores NOT applied. Alignment noise "
              "will be high for mined/comparable sources.", file=sys.stderr)

    # ---- 3a. tidy localisation strings, then drop pairs that became identical
    tidy_src = {x.strip() for x in a.tidy_sources.split(",") if x.strip()}
    if tidy_src:
        seen2, tidy_rows = set(), []
        for r in rows:
            if r["source"] in tidy_src:
                r["kk"], r["zh"] = tidy_ui(r["kk"], r["zh"])
            k2 = (r["kk"].lower(), r["zh"])
            if k2 in seen2:
                drops["exact_duplicate"] += 1
                continue
            seen2.add(k2)
            tidy_rows.append(r)
        rows = tidy_rows

    # ---- 3b. cap sentences that follow one template (stub articles: "X is a municipality in Y ...")
    if a.template_cap > 0 and strict_src:
        groups = collections.defaultdict(list)
        for idx, r in enumerate(rows):
            if r["source"] in strict_src:
                k = template_key(r["kk"])
                if k:
                    groups[k].append(idx)
        drop_idx = set()
        for k, idxs in groups.items():
            if len(idxs) > a.template_cap:
                idxs.sort(key=lambda j: (-rows[j].get("labse", 0.0),
                                         hashlib.sha256(f"{a.seed}:{rows[j]['kk']}".encode()).hexdigest()))
                drop_idx.update(idxs[a.template_cap:])
        if drop_idx:
            drops["template_cap"] += len(drop_idx)
            rows = [r for j, r in enumerate(rows) if j not in drop_idx]
        stats["after_template_cap"] = len(rows)

    # ---- 4. near-duplicate dedup (pair level, char 5-gram MinHash)
    try:
        from datasketch import MinHashLSH
        lsh = MinHashLSH(threshold=a.near_dup, num_perm=128)
        kept = []
        for idx, r in enumerate(rows):
            m = minhash(r["kk"].lower() + "\u241e" + r["zh"])
            if lsh.query(m):
                drops["near_duplicate"] += 1
                continue
            lsh.insert(str(idx), m)
            kept.append(r)
        rows = kept
        stats["after_near_dedup"] = len(rows)
    except ImportError:
        print("WARNING: datasketch missing, near-duplicate removal skipped", file=sys.stderr)

    # ---- 5. document-level split
    by_doc = collections.defaultdict(list)
    for r in rows:
        by_doc[r["doc_id"]].append(r)
    order = sorted(by_doc, key=lambda d: hashlib.sha256(f"{a.seed}:{d}".encode()).hexdigest())
    split = {"train": [], "dev": [], "test": []}
    held_src = {x.strip() for x in a.heldout_sources.split(",") if x.strip()}
    for d in order:
        docrows = by_doc[d]
        ok = len(docrows) <= a.max_doc_pairs and \
            (not held_src or docrows[0]["source"] in held_src)
        if ok and len(split["test"]) < a.n_test:
            split["test"].extend(docrows)
        elif ok and len(split["dev"]) < a.n_dev:
            split["dev"].extend(docrows)
        else:
            split["train"].extend(docrows)
    excl = {x.strip() for x in a.exclude_from_train_sources.split(",") if x.strip()}
    if excl:
        keep_tr = [r for r in split["train"] if r["source"] not in excl]
        drops["train_source_excluded"] += len(split["train"]) - len(keep_tr)
        split["train"] = keep_tr
    if len(split["test"]) < a.n_test or len(split["dev"]) < a.n_dev:
        print(f"WARNING: could not fill dev/test (dev={len(split['dev'])}, "
              f"test={len(split['test'])}); eligible sources too small", file=sys.stderr)

    # ---- 6. leakage screen: remove from train anything resembling dev/test
    held = split["dev"] + split["test"]
    held_pairs = {(r["kk"].lower(), r["zh"]) for r in held}
    held_kk = {r["kk"].lower() for r in held}
    held_zh = {r["zh"] for r in held}
    held_ngrams = set()
    for r in held:
        held_ngrams |= ngram_keys(r["kk"], r["zh"], a.ngram)
    try:
        from datasketch import MinHashLSH
        hl = MinHashLSH(threshold=a.leak_jaccard, num_perm=128)
        for j, r in enumerate(held):
            hl.insert(str(j), minhash(r["kk"].lower() + "\u241e" + r["zh"]))
    except ImportError:
        hl = None
    clean = []
    for r in split["train"]:
        if (r["kk"].lower(), r["zh"]) in held_pairs:
            drops["train_exact_in_heldout"] += 1
        elif r["kk"].lower() in held_kk or r["zh"] in held_zh:
            drops["train_same_side_as_heldout"] += 1
        elif ngram_keys(r["kk"], r["zh"], a.ngram) & held_ngrams:
            drops["train_8gram_overlap_heldout"] += 1
        elif hl is not None and hl.query(minhash(r["kk"].lower() + "\u241e" + r["zh"])):
            drops["train_nearDup_heldout"] += 1
        else:
            clean.append(r)
    split["train"] = clean

    # ---- 7. write
    os.makedirs(a.outdir, exist_ok=True)
    cols = ["id", "kk", "zh", "script", "source", "doc_id", "license", "url"]
    if a.labse:
        cols.append("labse")
    files = []
    for name, rs in split.items():
        for r in rs:
            r["id"] = pair_id(r["kk"], r["zh"])
        p = os.path.join(a.outdir, f"{name}.tsv")
        write_tsv(p, rs, cols)
        files.append(p)
    final = [r for rs in split.values() for r in rs]
    manifest = dict(
        created=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        config={k: v for k, v in vars(a).items() if k != "func"},
        stages=stats,
        dropped_by_reason=dict(drops.most_common()),
        split_sizes={k: len(v) for k, v in split.items()},
        by_source=dict(collections.Counter(r["source"] for r in final).most_common()),
        by_kk_script=dict(collections.Counter(r["script"] for r in final)),
        kk_words_per_zh_char_percentiles_before_ratio_filter=ratio_pct,
        licences=dict(collections.Counter(r["license"] or "UNSPECIFIED" for r in final)),
    )
    if a.dump_rejected:
        items = [it for rs in rejected.values() for it in rs]
        write_tsv(a.dump_rejected, items, ["reason", "source", "labse", "kk", "zh"])
        print(f"{len(items)} rejected pairs sampled -> {a.dump_rejected}", file=sys.stderr)
    mp = os.path.join(a.outdir, "manifest.json")
    with open(mp, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    files.append(mp)
    with open(os.path.join(a.outdir, "CHECKSUMS.sha256"), "w") as f:
        for p in files:
            f.write(f"{sha256_file(p)}  {os.path.basename(p)}\n")
    print(json.dumps({"split_sizes": manifest["split_sizes"], "stages": stats,
                      "dropped": manifest["dropped_by_reason"]},
                     ensure_ascii=False, indent=2))


# --------------------------------------------------------------- other tools
def cmd_moses2tsv(a):
    with open(a.kk, encoding="utf-8") as fk, open(a.zh, encoding="utf-8") as fz, \
            open(a.out, "w", encoding="utf-8", newline="") as fo:
        fo.write("kk\tzh\tsource\tdoc_id\tlicense\turl\n")
        n = 0
        for i, (k, z) in enumerate(zip(fk, fz)):
            k, z = normalise(k), normalise(z)
            fo.write(f"{k}\t{z}\t{a.source}\t{a.source}#chunk{i // 50}\t{a.license}\t\n")
            n += 1
    print(f"{n} lines -> {a.out}")


def cmd_audit(a):
    rows = list(read_tsv(a.input))
    random.Random(a.seed).shuffle(rows)
    if a.labse_bins:
        edges = [float(x) for x in a.labse_bins.split(",")] + [float("inf")]
        if not rows or "labse" not in rows[0]:
            sys.exit("--labse-bins needs an input with a 'labse' column (build with --labse)")
        bins = collections.defaultdict(list)
        for r in rows:
            v = float(r["labse"])
            for lo, hi in zip(edges, edges[1:]):
                if lo <= v < hi:
                    r["bin"] = f"{lo:.2f}-{hi:.2f}" if hi != float("inf") else f"{lo:.2f}+"
                    bins[r["bin"]].append(r)
                    break
        sample = []
        for name in sorted(bins):
            take = bins[name][:a.per_bin]
            print(f"  bin {name:<11} available={len(bins[name]):>6}  sampled={len(take)}")
            sample.extend(take)
        random.Random(a.seed + 1).shuffle(sample)     # do not present the bins in order
        for r in sample:
            r.update(label_a="", label_b="", label_final="", note="")
        write_tsv(a.out, sample, ["id", "kk", "zh", "source", "script", "labse", "bin",
                                  "label_a", "label_b", "label_final", "note"])
        print(f"{len(sample)} pairs -> {a.out}\nLabels: p = parallel, h = partially parallel, n = not parallel")
        return
    by_src = collections.defaultdict(list)
    for r in rows:
        by_src[r.get("source", "unknown")].append(r)
    sample = []
    for src, rs in by_src.items():
        k = a.per_source if a.per_source else max(1, round(a.n * len(rs) / len(rows)))
        sample.extend(rs[:k])
    if not a.per_source:
        sample = sample[:a.n]
    for r in sample:
        r.update(label_a="", label_b="", label_final="")
    write_tsv(a.out, sample, ["id", "kk", "zh", "source", "script",
                              "label_a", "label_b", "label_final"])
    print(f"{len(sample)} pairs -> {a.out}\nLabels: p = parallel, h = partially parallel, n = not parallel")


def cmd_kappa(a):
    allrows = list(read_tsv(a.input))

    def resolved(r):
        if r.get("label_final"):
            return r["label_final"]
        la, lb = r.get("label_a", ""), r.get("label_b", "")
        if la and lb:
            return la if la == lb else ""
        return la or lb                              # a single labeller: take that label

    both = [r for r in allrows if r.get("label_a") and r.get("label_b")]
    if both:
        labels = sorted({r["label_a"] for r in both} | {r["label_b"] for r in both})
        n = len(both)
        po = sum(r["label_a"] == r["label_b"] for r in both) / n
        ca = collections.Counter(r["label_a"] for r in both)
        cb = collections.Counter(r["label_b"] for r in both)
        pe = sum(ca[l] * cb[l] for l in labels) / (n * n)
        kappa = (po - pe) / (1 - pe) if pe < 1 else 1.0
        print(f"two labellers, n={n}: observed agreement={po:.3f}  Cohen's kappa={kappa:.3f}")
    else:
        print("only one labeller found: kappa is NOT computed (do not report an agreement figure)")
    lab = [(r, resolved(r)) for r in allrows]
    lab = [(r, x) for r, x in lab if x]
    if not lab:
        sys.exit("no labelled rows")
    unresolved = sum(1 for r in allrows if (r.get("label_a") or r.get("label_b")) and not resolved(r))
    print(f"labelled={len(lab)}  unresolved disagreements={unresolved}")

    def table(title, keyf):
        groups = collections.defaultdict(list)
        for r, x in lab:
            groups[keyf(r)].append(x)
        print(f"\n{title}")
        print(f"  {'':<26}{'N':>5}{'p':>8}{'h':>7}{'n':>7}   share p  (95% CI, Wilson)")
        for k in sorted(groups):
            g = groups[k]; m = len(g); c = collections.Counter(g)
            ph = c["p"] / m
            z = 1.96
            den = 1 + z * z / m
            ctr = (ph + z * z / (2 * m)) / den
            half = z * math.sqrt(ph * (1 - ph) / m + z * z / (4 * m * m)) / den
            print(f"  {k:<26}{m:>5}{c['p']:>8}{c['h']:>7}{c['n']:>7}   {ph:6.2f}   [{max(0.0, ctr - half):.2f}, {min(1.0, ctr + half):.2f}]")

    table("by source", lambda r: r.get("source", "?"))
    if "bin" in allrows[0]:
        table("by LaBSE bin (use this to choose tau)", lambda r: r.get("bin", "?"))
    tot = collections.Counter(x for _, x in lab)
    print(f"\noverall: p={tot['p']}  h={tot['h']}  n={tot['n']}  of {len(lab)}")


def main(argv=None):
    if argv is None and any(k in sys.argv[0] for k in ("ipykernel", "colab_kernel_launcher")):
        print("Notebook detected: sys.argv belongs to the Jupyter kernel, not to this script.\n"
              "Either save this file and run it as a command, e.g.\n"
              "    !python build_corpus.py <command> --options\n"
              "or call it from a cell with an explicit argument list, e.g.\n"
              "    main(['build', '--input', 'all.tsv', '--outdir', 'out'])")
        return

    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)

    m = sub.add_parser("moses2tsv")
    m.add_argument("--kk", required=True); m.add_argument("--zh", required=True)
    m.add_argument("--out", required=True); m.add_argument("--source", required=True)
    m.add_argument("--license", default="")
    m.set_defaults(func=cmd_moses2tsv)

    b = sub.add_parser("build")
    b.add_argument("--input", required=True); b.add_argument("--outdir", required=True)
    b.add_argument("--seed", type=int, default=13)
    b.add_argument("--n-dev", type=int, default=2000); b.add_argument("--n-test", type=int, default=2000)
    b.add_argument("--heldout-sources", default="",
                   help="comma-separated source names (e.g. opus:TED2020,opus:News-Commentary); "
                        "if set, dev/test are drawn ONLY from these sources")
    b.add_argument("--exclude-from-train-sources", default="",
                   help="comma-separated sources removed from TRAIN after the split. Use for the sources "
                        "dev/test come from (and their near-copies, e.g. opus:NeuLab-TedTalks): doc_id in "
                        "OPUS is only a 50-line chunk, so the same talk/article could otherwise sit on both sides")
    b.add_argument("--dump-rejected", default="",
                   help="write a random sample of dropped pairs (per reason) to this TSV, for checking the filters")
    b.add_argument("--dump-k", type=int, default=100, help="pairs per reason for --dump-rejected")
    b.add_argument("--max-doc-pairs", type=int, default=500,
                   help="documents larger than this are never used for dev/test")
    b.add_argument("--min-kk-words", type=int, default=2); b.add_argument("--max-kk-words", type=int, default=100)
    b.add_argument("--min-zh-chars", type=int, default=3); b.add_argument("--max-zh-chars", type=int, default=250)
    b.add_argument("--ratio-min", type=float, default=0.20); b.add_argument("--ratio-max", type=float, default=2.8)
    b.add_argument("--kk-script-share", type=float, default=0.6)
    b.add_argument("--kk-latin", choices=["diacritics", "drop", "keep"], default="diacritics",
                   help="Kazakh side written in Latin letters: 'diacritics' keeps it only if it contains "
                        "Kazakh Latin letters (otherwise it is English), 'drop' removes all, 'keep' keeps all")
    b.add_argument("--strict-sources", default="wikipedia:kk-zh",
                   help="sources that are comparable, not parallel: extra rules (number consistency, "
                        "headings) and a cap per sentence template; empty = off")
    b.add_argument("--tidy-sources", default="opus:KDE4,opus:GNOME,opus:Ubuntu",
                   help="sources whose strings get tokenisation/accelerator clean-up (empty = off)")
    b.add_argument("--strict-tau", type=float, default=None,
                   help="LaBSE threshold for --strict-sources only (default: same as --tau)")
    b.add_argument("--template-cap", type=int, default=300,
                   help="max sentences per name-masked template within --strict-sources (0 = no cap)")
    b.add_argument("--zh-strip-english-sources", default="opus:OpenSubtitles",
                   help="sources whose zh side may carry an appended English subtitle line; the English "
                        "run is removed when the line would otherwise be dropped (empty = never)")
    b.add_argument("--keep-spam", action="store_true",
                   help="do not drop adult/gambling spam (default: dropped, see SPAM_KK / SPAM_ZH)")
    b.add_argument("--zh-nonhan-share", type=float, default=0.3)
    b.add_argument("--max-digit-share", type=float, default=0.5)
    b.add_argument("--zh-script", choices=["drop_traditional", "to_simplified", "keep"],
                   default="to_simplified",
                   help="to_simplified converts traditional "
                        "to simplified (written Cantonese is always dropped); drop_traditional drops it")
    b.add_argument("--keep-russian", action="store_true", help="disable the likely-Russian heuristic")
    b.add_argument("--labse", action="store_true"); b.add_argument("--labse-model", default="sentence-transformers/LaBSE")
    b.add_argument("--labse-batch", type=int, default=4096)
    b.add_argument("--labse-cache", default="",
                   help="file that stores LaBSE scores; makes reruns and --tau changes instant "
                        "and lets an interrupted run resume")
    b.add_argument("--tau", type=float, default=0.72)
    b.add_argument("--arabic-policy", choices=["score", "keep_unscored", "drop"], default="score")
    b.add_argument("--near-dup", type=float, default=0.85)
    b.add_argument("--leak-jaccard", type=float, default=0.7)
    b.add_argument("--ngram", type=int, default=8)
    b.set_defaults(func=cmd_build)

    au = sub.add_parser("audit")
    au.add_argument("--input", required=True); au.add_argument("--out", required=True)
    au.add_argument("--labse-bins", default="",
                    help="comma-separated lower edges, e.g. 0.5,0.6,0.72,0.8,0.9; samples --per-bin pairs "
                         "from each LaBSE-score bin (use it on a build made with --tau -1 to calibrate tau)")
    au.add_argument("--per-bin", type=int, default=100)
    au.add_argument("--per-source", type=int, default=0,
                    help="take this many pairs from EVERY source instead of proportional sampling")
    au.add_argument("--n", type=int, default=500); au.add_argument("--seed", type=int, default=13)
    au.set_defaults(func=cmd_audit)

    k = sub.add_parser("kappa")
    k.add_argument("--input", required=True)
    k.set_defaults(func=cmd_kappa)

    a = ap.parse_args(argv)
    a.func(a)


if __name__ == "__main__":
    main()


In [ ]:
# 4. Данные OPUS (скачиваются один раз и остаются на Диске)
want = SKIP_DOWNLOAD + ('|trad' if ADD_TRADITIONAL else '')          # что именно должно лежать в opus_all.tsv
marker = f'{OPUS_DIR}/.skip'
old_want = open(marker, encoding='utf-8').read() if os.path.exists(marker) else None
have = os.path.exists(OPUS_ALL) and os.path.getsize(OPUS_ALL) > 0
if have and old_want is None:
    old_want = 'KDE4,GNOME,Ubuntu,XLEnt'              # так скачивалось в прежних версиях ноутбука
if have and not FORCE_DOWNLOAD and old_want == want:
    n = sum(1 for _ in open(OPUS_ALL, encoding='utf-8')) - 1
    print(f'OPUS уже на Диске ({n} пар), скачивание пропущено: {OPUS_ALL}')
else:
    sh([sys.executable, '/content/fetch_opus.py', 'download', '--out', OPUS_DIR]
       + ([] if ADD_TRADITIONAL else ['--no-traditional']) + ['--skip', SKIP_DOWNLOAD])
    os.makedirs(OPUS_DIR, exist_ok=True)
    open(marker, 'w', encoding='utf-8').write(want)


### Википедия (второй источник)
Пары предложений из статей казахской и китайской Википедии, связанных межъязыковыми ссылками. Лицензия CC BY-SA 4.0 допускает публикацию корпуса (с указанием источника: ссылка на статью лежит в колонке `url`).

In [ ]:
%%writefile /content/mine_wikipedia.py
#!/usr/bin/env python3
"""
Mine Kazakh-Chinese sentence pairs from Wikipedia.

Idea (the same as WikiMatrix, simplified): take every Kazakh article that has an interlanguage link
to a Chinese article, split both into sentences, embed the sentences with LaBSE and keep the pairs
that are each other's nearest neighbour INSIDE the article pair. The result is a candidate list;
build_corpus.py then applies its usual filters and the LaBSE threshold tau.

Wikipedia text is licensed CC BY-SA 4.0 (attribution + share-alike), which allows redistribution.
Every output row carries the Kazakh article URL in the 'url' column for attribution.

Stages (both are resumable; rerunning continues where it stopped)
  prepare  fetch the interlanguage links (MediaWiki API), stream the Kazakh and Chinese articles, write articles.jsonl.gz
  mine     embed the sentences, match them, append to wiki_pairs.tsv

Article sources (--kk-articles / --zh-articles):
  hf:wikimedia/wikipedia:20231101.kk     Hugging Face dataset  (columns id, url, title, text)
  path/to/file.jsonl                      local JSON lines with the same fields (used for tests)
"""
import argparse
import gzip
import io
import json
import os
import re
import sys
import time
import urllib.error
import urllib.parse
import urllib.request

HAN = re.compile(r"[㐀-䶿一-鿿豈-﫿]")
CYR = re.compile(r"[Ѐ-ӿ]")
LETTER = re.compile(r"[^\W\d_]", re.U)
KK_SPLIT = re.compile(r"(?<=[.!?…])\s+(?=[«\"“(A-ZА-ЯӘІҢҒҮҰҚӨҺ0-9])")
ZH_SPLIT = re.compile(r"(?<=[。！？；])")
LANGLINK = re.compile(r"\((\d+),'zh','((?:[^'\\]|\\.)*)'\)")
WS = re.compile(r"\s+")


# ------------------------------------------------------------------ sentences
def clean(s):
    return WS.sub(" ", s).strip()


def kk_sentences(text, a):
    out = []
    for para in text.split("\n"):
        for s in KK_SPLIT.split(para):
            s = clean(s)
            w = len(s.split())
            if not (a.kk_min_words <= w <= a.kk_max_words):
                continue
            letters = LETTER.findall(s)
            if not letters or len(CYR.findall(s)) / len(letters) < 0.7:
                continue                      # mostly Latin/Arabic/other: not Kazakh Cyrillic prose
            out.append(s)
    return out[: a.max_sents]


def zh_sentences(text, a):
    out = []
    for para in text.split("\n"):
        for s in ZH_SPLIT.split(para):
            s = clean(s)
            n = len(HAN.findall(s))
            if not (a.zh_min_chars <= n and len(s) <= a.zh_max_chars):
                continue
            if n / max(1, len(s)) < 0.5:
                continue
            out.append(s)
    return out[: a.max_sents]


# ------------------------------------------------------------------ input
def open_maybe_gz(path_or_url):
    if path_or_url.startswith(("http://", "https://")):
        req = urllib.request.Request(path_or_url, headers={"User-Agent": "kkzh-corpus/0.1 (research)"})
        raw = urllib.request.urlopen(req, timeout=120)
        return gzip.open(raw, "rb") if path_or_url.endswith(".gz") else raw
    return gzip.open(path_or_url, "rb") if path_or_url.endswith(".gz") else open(path_or_url, "rb")


def read_langlinks(src):
    """Return {kk_page_id: zh_title} from a MediaWiki langlinks SQL dump."""
    links = {}
    with open_maybe_gz(src) as f:
        for line in io.TextIOWrapper(f, encoding="utf-8", errors="replace"):
            if not line.startswith("INSERT INTO"):
                continue
            for m in LANGLINK.finditer(line):
                title = m.group(2).replace("\\'", "'").replace('\\"', '"').replace("\\\\", "\\")
                links[int(m.group(1))] = title.replace("_", " ")
    return links


USER_AGENT = "kkzh-corpus/0.1 (research corpus builder)"


def http_json(url, tries=10):
    """GET a JSON document. Wikimedia throttles shared cloud IPs: on 429/503 wait for Retry-After
    (or back off exponentially, at most 2 minutes) and try again."""
    for k in range(tries):
        wait = min(2 ** k, 120)
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
            with urllib.request.urlopen(req, timeout=60) as r:
                return json.loads(r.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            ra = e.headers.get("Retry-After") if e.headers else None
            if ra and ra.isdigit():
                wait = min(int(ra) + 1, 300)
            print(f"  HTTP {e.code}; retry in {wait}s", flush=True)
        except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as e:
            print(f"  request failed ({e}); retry in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError("Wikipedia API did not answer: " + url[:120] +
                       "\nRun the cell again later: it resumes from where it stopped.")


def fetch_langlinks_api(outdir, a):
    """{kk title: zh title} through the MediaWiki API (generator=allpages, prop=langlinks).
    Resumable: pairs are appended to langlinks_api.tsv and the continuation token is stored."""
    path = os.path.join(outdir, "langlinks_api.tsv")
    state, done_flag = path + ".state", path + ".done"
    links = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                k, _, z = line.rstrip("\n").partition("\t")
                if z:
                    links[k] = z
    if os.path.exists(done_flag):
        return links
    cont = json.load(open(state)) if os.path.exists(state) else {}
    params = dict(action="query", format="json", formatversion="2", generator="allpages", gaplimit="500",
                  gapfilterredir="nonredirects", gapnamespace="0", prop="langlinks", lllang="zh",
                  lllimit="500")
    n = 0
    with open(path, "a", encoding="utf-8") as f:
        while True:
            data = http_json(a.api_url + "?" + urllib.parse.urlencode({**params, **cont}))
            for pg in data.get("query", {}).get("pages", []):
                for ll in pg.get("langlinks", []):
                    f.write(f"{pg['title']}\t{ll['title']}\n")
                    links[pg["title"]] = ll["title"]
            f.flush()
            if "continue" not in data:
                break
            cont = data["continue"]
            json.dump(cont, open(state, "w"))
            n += 1
            if n % 20 == 0:
                print(f"  API requests {n}, links so far {len(links)}, at {cont.get('gapcontinue', '?')[:30]}",
                      flush=True)
            time.sleep(a.api_delay)
    open(done_flag, "w").write("1")
    return links


def iter_articles(spec):
    if spec.startswith("hf:"):
        _, repo, config = spec.split(":", 2)
        from datasets import load_dataset
        yield from load_dataset(repo, config, split="train", streaming=True)
    else:
        with open(spec, encoding="utf-8") as f:
            for line in f:
                yield json.loads(line)


# ------------------------------------------------------------------ prepare
def cmd_prepare(a):
    global USER_AGENT
    if a.contact:
        USER_AGENT = f"kkzh-corpus/0.1 (research corpus builder; {a.contact})"
    os.makedirs(a.outdir, exist_ok=True)
    target = os.path.join(a.outdir, "articles.jsonl.gz")
    if os.path.exists(target) and os.path.getsize(target) > 100:      # an empty gzip is ~20 bytes
        print("articles.jsonl.gz exists, prepare skipped:", target)
        return
    t0 = time.time()
    print("reading interlanguage links ...", flush=True)
    if a.langlinks == "api":
        links = fetch_langlinks_api(a.outdir, a)                 # keys: Kazakh titles
    else:
        links = read_langlinks(a.langlinks)                      # keys: Kazakh page ids
    print(f"  {len(links)} Kazakh pages with a Chinese interlanguage link ({time.time() - t0:.0f}s)")
    if not links:
        sys.exit("no interlanguage links found: stopping (nothing was written)")

    # 1) Kazakh side: keep linked articles that have enough sentences
    kk = {}
    for i, art in enumerate(iter_articles(a.kk_articles)):
        zt = links.get(art["title"].replace("_", " "))
        if zt is None:
            zt = links.get(int(art["id"]))
        if zt is None:
            continue
        sents = kk_sentences(art["text"], a)
        if len(sents) >= a.min_sents:
            kk[zt] = dict(title=art["title"], url=art.get("url", ""), kk=sents)
        if (i + 1) % 50000 == 0:
            print(f"  kk articles scanned: {i + 1}, kept {len(kk)}", flush=True)
    print(f"Kazakh articles kept: {len(kk)}")
    if not kk:
        sys.exit("no Kazakh article matched the links (check --kk-articles): stopping (nothing was written)")

    # 2) Chinese side: only the titles we need
    tmp = target + ".part"
    n = 0
    with gzip.open(tmp, "wt", encoding="utf-8") as out:
        for i, art in enumerate(iter_articles(a.zh_articles)):
            rec = kk.get(art["title"].replace("_", " "))
            if rec is None:
                continue
            zs = zh_sentences(art["text"], a)
            if len(zs) < a.min_sents:
                continue
            rec = dict(rec, zh=zs)
            out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n += 1
            if (i + 1) % 100000 == 0:
                print(f"  zh articles scanned: {i + 1}, matched {n}", flush=True)
    os.replace(tmp, target)
    print(f"article pairs written: {n} -> {target} ({time.time() - t0:.0f}s)")


# ------------------------------------------------------------------ mine
def cmd_mine(a):
    import numpy as np
    from sentence_transformers import SentenceTransformer

    src = os.path.join(a.outdir, "articles.jsonl.gz")
    if not os.path.exists(src):
        sys.exit("run 'prepare' first")
    out_path = os.path.join(a.outdir, "wiki_pairs.tsv")
    prog_path = os.path.join(a.outdir, "progress.txt")
    done = int(open(prog_path).read().strip() or 0) if os.path.exists(prog_path) else 0
    if done == 0 or not os.path.exists(out_path):
        done = 0
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("kk\tzh\tsource\tdoc_id\tlicense\turl\tcos\n")
    print(f"resuming after {done} article pairs" if done else "starting from the beginning", flush=True)

    model = SentenceTransformer(a.model)
    try:
        import torch
        if torch.cuda.is_available():
            model.half()
    except Exception:
        pass

    def embed(texts):
        return np.asarray(model.encode(texts, batch_size=a.batch, normalize_embeddings=True,
                                       convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)

    def flush(batch, out):
        kk_all = [s for r in batch for s in r["kk"]]
        zh_all = [s for r in batch for s in r["zh"]]
        ek, ez = embed(kk_all), embed(zh_all)
        ik = iz = 0
        kept = 0
        for r in batch:
            m, n = len(r["kk"]), len(r["zh"])
            S = ek[ik:ik + m] @ ez[iz:iz + n].T
            ik += m
            iz += n
            best_j = S.argmax(axis=1)
            best_i = S.argmax(axis=0)
            for i in range(m):
                j = int(best_j[i])
                if int(best_i[j]) != i or S[i, j] < a.min_cos:
                    continue
                row = [r["kk"][i], r["zh"][j], "wikipedia:kk-zh", "wiki:" + r["title"],
                       "CC BY-SA 4.0 (Wikipedia)", r["url"], f"{S[i, j]:.4f}"]
                out.write("\t".join(x.replace("\t", " ").replace("\n", " ") for x in row) + "\n")
                kept += 1
        return kept

    t0 = time.time()
    total, seen, batch = 0, 0, []
    with gzip.open(src, "rt", encoding="utf-8") as f, open(out_path, "a", encoding="utf-8") as out:
        for line in f:
            seen += 1
            if seen <= done:
                continue
            batch.append(json.loads(line))
            if len(batch) >= a.articles_per_batch:
                total += flush(batch, out)
                out.flush()
                open(prog_path, "w").write(str(seen))
                print(f"  article pairs {seen}, pairs kept in this run {total}, "
                      f"{time.time() - t0:.0f}s", flush=True)
                batch = []
        if batch:
            total += flush(batch, out)
            out.flush()
            open(prog_path, "w").write(str(seen))
    print(f"done: {seen} article pairs, {total} pairs added in this run -> {out_path}")


def main(argv=None):
    if argv is None and any(k in sys.argv[0] for k in ("ipykernel", "colab_kernel_launcher")):
        print("Notebook detected: run as a command, e.g.  !python mine_wikipedia.py prepare --outdir ...")
        return
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    for name, fn in (("prepare", cmd_prepare), ("mine", cmd_mine)):
        p = sub.add_parser(name)
        p.add_argument("--outdir", required=True)
        p.add_argument("--langlinks", default="api",
                       help="'api' (MediaWiki API, default) or a langlinks SQL dump (path or URL, .sql.gz)")
        p.add_argument("--api-url", default="https://kk.wikipedia.org/w/api.php")
        p.add_argument("--api-delay", type=float, default=1.0, help="seconds between API requests")
        p.add_argument("--contact", default="",
                       help="optional contact (e-mail or URL) added to the User-Agent, as Wikimedia asks")
        p.add_argument("--kk-articles", default="hf:wikimedia/wikipedia:20231101.kk")
        p.add_argument("--zh-articles", default="hf:wikimedia/wikipedia:20231101.zh")
        p.add_argument("--kk-min-words", type=int, default=4); p.add_argument("--kk-max-words", type=int, default=80)
        p.add_argument("--zh-min-chars", type=int, default=6); p.add_argument("--zh-max-chars", type=int, default=250)
        p.add_argument("--max-sents", type=int, default=300, help="sentences kept per article and side")
        p.add_argument("--min-sents", type=int, default=3)
        p.add_argument("--model", default="sentence-transformers/LaBSE")
        p.add_argument("--batch", type=int, default=256)
        p.add_argument("--articles-per-batch", type=int, default=200)
        p.add_argument("--min-cos", type=float, default=0.55,
                       help="candidates below this cosine are not written; build_corpus.py applies the real tau")
        p.set_defaults(func=fn)
    a = ap.parse_args(argv)
    a.func(a)


if __name__ == "__main__":
    main()


In [ ]:
# 4b. Википедия: пары предложений из статей, связанных межъязыковыми ссылками (CC BY-SA 4.0)
# Занимает долго (скачивание дампа, чтение статей, LaBSE на GPU). Можно прервать и запустить снова:
# уже готовое не пересчитывается, майнинг продолжается с места остановки.
if USE_WIKI:
    for step in ('prepare', 'mine'):
        sh([sys.executable, '/content/mine_wikipedia.py', step, '--outdir', WIKI_DIR]
           + (['--contact', WIKI_CONTACT] if WIKI_CONTACT else []) + WIKI_EXTRA_ARGS)
else:
    print('Википедия выключена (USE_WIKI = False)')


In [ ]:
%%writefile /content/fetch_weblate.py
#!/usr/bin/env python3
"""Kazakh-Chinese pairs from ayymen/Weblate-Translations by pivoting through the English source string.

The dataset has no kk-zh configuration; it has en-kk and en-zh_Hans (and variants). When the SAME English
source string was translated into both languages, the two translations are a kk-zh pair of one source.
Strings with more than one distinct translation on either side are skipped (the English string is
ambiguous: different senses, different contexts), as are untranslated strings.
The licence of every row is kept (columns kk:<licence>|zh:<licence>); check them before publishing.

    python fetch_weblate.py --out weblate/weblate_pairs.tsv
"""
import argparse
import collections
import csv
import os
import sys

REPO = "ayymen/Weblate-Translations"
KK_CONFIGS = ["en-kk", "en_GB-kk", "en-kk_KZ"]
ZH_CONFIGS = ["en-zh_Hans", "en-zh-Hans", "en-zh", "en_US-zh_Hans", "en_devel-zh_Hans"]
ZH_TRAD_CONFIGS = ["en-zh_Hant", "en-zh-Hant", "en_US-zh_Hant", "en_GB-zh_Hant", "en_devel-zh_Hant"]


def norm(s):
    return " ".join(str(s or "").split())


def collect(rows):
    """rows: iterable of (english, target, licence) -> {english: {target: licence}}"""
    out = collections.defaultdict(dict)
    for en, tgt, lic in rows:
        en, tgt = norm(en), norm(tgt)
        if not en or not tgt or en == tgt:
            continue
        out[en].setdefault(tgt, norm(lic) or "unknown")
    return out


def pivot(kk, zh):
    """Join on the English string; keep only unambiguous strings. Returns (pairs, stats)."""
    pairs, amb, common = [], 0, 0
    for en in sorted(set(kk) & set(zh)):
        common += 1
        if len(kk[en]) != 1 or len(zh[en]) != 1:
            amb += 1
            continue
        (k, kl), = kk[en].items()
        (z, zl), = zh[en].items()
        pairs.append((k, z, f"kk:{kl}|zh:{zl}"))
    return pairs, dict(english_strings_in_both=common, ambiguous_skipped=amb, pairs=len(pairs))


def load_configs(configs, repo, tag):
    from datasets import load_dataset
    rows = []
    for cfg in configs:
        try:
            ds = load_dataset(repo, cfg, split="train")
        except Exception as e:                       # config absent or unreadable
            print(f"  [skip] {tag} config {cfg}: {type(e).__name__}", file=sys.stderr)
            continue
        n0 = len(rows)
        for r in ds:
            rows.append((r["source_string"], r["target_string"], r.get("license", "")))
        print(f"  {tag} {cfg}: {len(rows) - n0} rows", file=sys.stderr)
    return rows


def main(argv=None):
    if argv is None and any(k in sys.argv[0] for k in ("ipykernel", "colab_kernel_launcher")):
        print("Notebook detected: call main([...]) with an explicit argument list or run as a command.")
        return
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out", required=True)
    ap.add_argument("--repo", default=REPO)
    ap.add_argument("--traditional", action="store_true",
                    help="also use the Traditional-Chinese configs (converted to Simplified later in the build)")
    a = ap.parse_args(argv)
    kk = collect(load_configs(KK_CONFIGS, a.repo, "kk"))
    zh = collect(load_configs(ZH_CONFIGS + (ZH_TRAD_CONFIGS if a.traditional else []), a.repo, "zh"))
    if not kk or not zh:
        sys.exit("No Kazakh or no Chinese rows could be loaded (dataset name changed or no network).")
    pairs, st = pivot(kk, zh)
    print("  " + ", ".join(f"{k}={v}" for k, v in st.items()), file=sys.stderr)
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    with open(a.out, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_NONE, quotechar=None, lineterminator="\n")
        w.writerow(["kk", "zh", "source", "doc_id", "license", "url"])
        for i, (k, z, lic) in enumerate(pairs):
            w.writerow([k, z, "weblate:en-pivot", f"weblate#chunk{i // 50}", lic,
                        "https://huggingface.co/datasets/" + a.repo])
    print(f"{len(pairs)} pairs -> {a.out}")


if __name__ == "__main__":
    main()


In [ ]:
# 4b2. Weblate-Translations (Hugging Face): пары kk-zh через общую английскую строку
if USE_WEBLATE:
    out = f'{WEBLATE_DIR}/weblate_pairs.tsv'
    if os.path.exists(out) and os.path.getsize(out) > 0 and not FORCE_DOWNLOAD:
        n = sum(1 for _ in open(out, encoding='utf-8')) - 1
        print(f'Weblate уже на Диске ({n} пар): {out}')
    else:
        sh([sys.executable, '/content/fetch_weblate.py', '--out', out] + (['--traditional'] if WEBLATE_TRADITIONAL else []))
else:
    print('Weblate выключен (USE_WEBLATE = False)')


In [ ]:
# 4c. Общий входной файл для сборки: OPUS (+ Википедия)
parts = [OPUS_ALL] + ([f'{WIKI_DIR}/wiki_pairs.tsv'] if USE_WIKI else []) + ([f'{WEBLATE_DIR}/weblate_pairs.tsv'] if USE_WEBLATE else [])
stamp = f'min_cos={WIKI_MIN_COS if USE_WIKI else None};weblate={USE_WEBLATE}'
marker = ALL_SRC + '.cfg'
old_stamp = open(marker, encoding='utf-8').read() if os.path.exists(marker) else ''
need = (not os.path.exists(ALL_SRC)) or old_stamp != stamp or \
    any(os.path.getmtime(p) > os.path.getmtime(ALL_SRC) for p in parts)
if need:
    n = skipped = 0
    with open(ALL_SRC + '.part', 'w', encoding='utf-8') as out:
        out.write('kk\tzh\tsource\tdoc_id\tlicense\turl\n')
        for p in parts:
            is_wiki = p.endswith('wiki_pairs.tsv')
            with open(p, encoding='utf-8') as f:
                next(f)                                  # заголовок
                for line in f:
                    cols = line.rstrip('\n').split('\t')
                    if is_wiki and len(cols) >= 7:
                        try:
                            if float(cols[6]) < WIKI_MIN_COS:
                                skipped += 1
                                continue
                        except ValueError:
                            pass
                    cols = cols[:6] + [''] * (6 - len(cols[:6]))
                    out.write('\t'.join(cols) + '\n')
                    n += 1
    os.replace(ALL_SRC + '.part', ALL_SRC)
    open(marker, 'w', encoding='utf-8').write(stamp)
    print(f'Собран {ALL_SRC}: {n} пар из {len(parts)} файлов (Википедия: отброшено с оценкой ниже {WIKI_MIN_COS}: {skipped})')
else:
    print('Общий входной файл уже актуален:', ALL_SRC)


In [ ]:
# 5. Сборка корпуса прямо в папку на Диске
if os.path.exists(f'{OUT}/train.tsv') and not OVERWRITE:
    print(f'Папка {OUT} уже собрана, сборка пропущена.')
    print('Чтобы собрать заново: поменяйте VERSION (новая папка) или поставьте OVERWRITE = True.')
else:
    cmd = [sys.executable, '/content/build_corpus.py', 'build',
           '--input', ALL_SRC, '--outdir', OUT,
           '--labse', '--labse-cache', CACHE,
           '--heldout-sources', HELDOUT,
           '--n-dev', str(N_DEV), '--n-test', str(N_TEST),
           '--tau', str(TAU), '--zh-script', ZH_SCRIPT,
           '--template-cap', str(TEMPLATE_CAP), '--strict-tau', str(WIKI_TAU)] + EXTRA_BUILD_ARGS
    if EXCLUDE_FROM_TRAIN:
        cmd += ['--exclude-from-train-sources', EXCLUDE_FROM_TRAIN]
    log = f'{BASE}/{VERSION}.build.log'
    sh(cmd, log=log)
    # копия кода и лога рядом с корпусом, чтобы версия была воспроизводима
    os.makedirs(f'{OUT}/code', exist_ok=True)
    for f in ('/content/build_corpus.py', '/content/fetch_opus.py', '/content/mine_wikipedia.py', '/content/fetch_weblate.py'):
        if os.path.exists(f):
            shutil.copy(f, f'{OUT}/code/')
    shutil.copy(log, f'{OUT}/build.log')
    print('\nГотово. Корпус сохранён в', OUT)


In [ ]:
# 6. Сводка и проверка контрольных сумм
m = json.load(open(f'{OUT}/manifest.json', encoding='utf-8'))
print('Размеры:', m['split_sizes'])
print('\nЭтапы:')
for k, v in m['stages'].items():
    print(f'  {k:<22}{v:>9}')
print('\nОтсеяно (по причинам):')
for k, v in list(m['dropped_by_reason'].items())[:12]:
    print(f'  {k:<30}{v:>9}')
print('\nПо источникам:', m['by_source'])
print('По письменности (kk):', m['by_kk_script'])
print('Лицензии:', m['licences'])

bad = 0
for line in open(f'{OUT}/CHECKSUMS.sha256', encoding='utf-8'):
    digest, name = line.split()
    h = hashlib.sha256()
    with open(f'{OUT}/{name}', 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    if h.hexdigest() != digest:
        bad += 1
        print('НЕ СОВПАДАЕТ хеш:', name)
print('\nКонтрольные суммы:', 'все совпали' if not bad else f'{bad} не совпали')

print('\nСодержимое папки:')
for root, _, files in os.walk(OUT):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        print(f'  {os.path.relpath(p, OUT):<24}{os.path.getsize(p) / 1e6:8.2f} MB')


### Диагностика качества (запускается один раз)
Следующая ячейка готовит два файла для проверки: случайные отброшенные пары по каждой причине (режут ли фильтры лишнее) и выборку пар из разных интервалов оценки LaBSE (чтобы выбрать порог `TAU` по измеренной точности, а не по числу из статьи).

In [ ]:
# 7. Диагностика: выборки для проверки фильтров и подбора порога TAU
# Запускается один раз. Оценки LaBSE берутся из кэша, поэтому быстро.
CALIB = f'{BASE}/{VERSION}_calibration'          # все пары без порога LaBSE (с баллами)
REJ = f'{BASE}/{VERSION}_rejected_sample.tsv'    # случайные отброшенные пары по каждой причине
AUDIT = f'{BASE}/{VERSION}_audit.tsv'            # выборка для разметки по интервалам LaBSE
PER_BIN = 80
BINS = '0.30,0.50,0.60,0.72,0.80,0.90'

if not RUN_DIAG:
    print('Диагностика выключена (RUN_DIAG = False), шаг пропущен.')
elif os.path.exists(AUDIT):
    print('Файлы диагностики уже есть, шаг пропущен:', AUDIT)
    print('Не перезаписываю: в этом файле может быть ваша разметка.')
else:
    sh([sys.executable, '/content/build_corpus.py', 'build',
        '--input', ALL_SRC, '--outdir', CALIB,
        '--labse', '--labse-cache', CACHE, '--tau', '-1',
        '--n-dev', '0', '--n-test', '0', '--zh-script', ZH_SCRIPT,
        '--dump-rejected', REJ, '--dump-k', '60'] + EXTRA_BUILD_ARGS)
    sh([sys.executable, '/content/build_corpus.py', 'audit',
        '--input', f'{CALIB}/train.tsv', '--out', AUDIT,
        '--labse-bins', BINS, '--per-bin', str(PER_BIN)])
    print('\nГотово. Пришлите мне два файла с Диска:')
    print(' ', REJ)
    print(' ', AUDIT)
